# 01 – Data Understanding

**Dataset:** track_history_data_august_2025.csv  
This notebook supports the Data Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Data overview

In [ ]:
df = pd.read_csv("../data_raw/transit_2026_raw.csv")
df.head(3)

#### Dimensions

In [ ]:
df.shape

#### Variables 

In [ ]:
df.columns

### 1.1 Standardizing column names and sorting

Column names are standardized by converting all characters to lowercase and replacing spaces with underscores.  
This ensures consistency and improves usability in subsequent data processing and analysis.

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [ ]:
df.columns

### 1.2 Temporal coverage

In [ ]:
pd.to_datetime(df["sd_date_string"], format="%d-%m-%Y").agg(["min", "max"])

In [ ]:
pd.to_datetime(df["sd_date_string"], format="%d-%m-%Y").dt.month.value_counts().sort_index()

In [ ]:
dates = pd.to_datetime(df["sd_date_string"], format="%d-%m-%Y")

print("Min date:", dates.min())
print("Max date:", dates.max())
print("Unique days:", dates.nunique())

### 1.3 Data types

In [ ]:
df.dtypes

The dataset contains numerical, categorical, and temporal variables.

Numerical variables (e.g., `distance_mtr`, `duration_sec`, `stay_duration_sec`) are correctly stored as `int64` and `float64` and are suitable for quantitative analysis.

Categorical variables are represented as strings and include classifications, transit rules, and location attributes (e.g., `primary_class`, `sl_room_local_name`).

Temporal variables (e.g., `sd_date_string`, `ed_date_string`, `st_time`, `et_time`) are currently stored as strings rather than datetime objects. While they contain relevant time information, they are not yet suitable for time-based analysis.

Conversion to datetime format is therefore deferred to the data preparation phase to ensure consistency after filtering and feature selection.

Additionally, the dataset contains multiple representations of temporal information (e.g., year, week, date string), indicating redundancy that will be addressed later in the pipeline.

## 2. Variable types

The dataset contains variables describing equipment identity, movement characteristics, spatial location, and time.

- **Equipment variables** identify equipment types and individual devices (e.g., `primary_class`, `secondary_class`, `tertiary_class`, `object_key`).

- **Movement variables** describe the characteristics of each transit event, including movement distance, duration, and time between movements (e.g., `distance_mtr`, `distance_floor`, `duration_sec`, `stay_duration_sec`, `segment_count`).

- **Location variables** represent the hierarchical location of equipment at the start and end of each movement, including building, floor, locality, room, and section levels (e.g., `sl_building_local_name`, `sl_room_local_name`, `el_building_local_name`, `el_room_local_name`).

- **Temporal variables** describe when transit events start and end, including date-related attributes and timestamps (e.g., `sd_date_string`, `ed_date_string`, `st_time`, `et_time`, as well as derived calendar variables such as `sd_week_of_year`).

Together, these variables enable analysis of equipment movements across devices, locations, and time.

## 3. Equipment variables

Identify equipment category hierarchy, identify indivdual devices and enable tracking of device histories.

In [ ]:
print(df["primary_class"].nunique())
print(df["secondary_class"].nunique())
print(df["tertiary_class"].nunique())

The dataset contains only a single equipment type (volumetric pumps), resulting in no variation in the equipment classification variables.

### 3.1 Unique devices
In addition, each device is uniquely identified by an object_key, which represents individual devices and enables analysis at the device level. The number of unique object keys represents the total number of individual devices in the dataset.

In [ ]:
# Number of unique devices
df["object_key"].nunique()

In [ ]:
# Number of observations per device
df["object_key"].value_counts().describe()

The number of observations per device varies considerably, with a median of 31 and a maximum of 345 observations.

This indicates substantial variation in device activity, where some devices appear much more frequently in the dataset than others.

## 4. Movement variables

**Variables**
- `distance_mtr`
- `distance_floor`
- `duration_sec`
- `stay_duration_sec`
- `segment_count`

**Purpose**  
These variables describe the characteristics of each transit event, including movement distance, duration, and time between movements. They form the basis for analyzing movement patterns and equipment utilization.

In [ ]:
pd.options.display.float_format = '{:,.0f}'.format
df[["distance_mtr", "duration_sec", "stay_duration_sec"]].describe()

In [ ]:
import matplotlib.pyplot as plt

cols = ["distance_mtr", "duration_sec", "stay_duration_sec"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(cols):
    values = df[col]
    values = values[values > 0]  # log kræver > 0
    
    axes[i].hist(values, bins=100)
    axes[i].set_xscale("log")
    
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("../figures/distributions_log.jpeg", dpi=300, bbox_inches='tight')
plt.show()

The movement variables show substantial variation across observations.

The average movement distance is approximately 31 meters, with a wide range from 0 to 975 meters, indicating both very short and long movements. Movement duration has a mean of 143 seconds and a median of 59 seconds, suggesting a right-skewed distribution with some longer movements.

The stay duration exhibits particularly high variability, with a mean of approximately 171,445 seconds and a maximum value exceeding 89 million seconds, indicating the presence of extreme values and long periods between movements.

In [ ]:
df["segment_count"].value_counts().plot(kind="bar")

The segment count is highly concentrated at low values, with most movements consisting of only a few segments.

The distribution is right-skewed, indicating that complex movements involving many segments are relatively rare.

In [ ]:
df["distance_floor"].value_counts().sort_index()

In [ ]:
import matplotlib.pyplot as plt

# Beregn fordeling
counts = df["distance_floor"].value_counts().sort_index()

# Plot
plt.figure(figsize=(10, 5))
plt.bar(counts.index, counts.values)

plt.title("Distribution of distance_floor")
plt.xlabel("distance_floor")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

The distance_floor variable is highly concentrated at low values, with the majority of movements occurring within the same floor.

Higher values are relatively rare, indicating that movements across multiple floors occur infrequently.

## 5. Temporal variables

**Variables**
- `sd_date_string`
- `ed_date_string`
- `st_time`
- `et_time`
- `sd_calendar_year`
- `sd_month_name`
- `sd_day_of_week_name`
- `sd_day_of_year`
- `sd_week_of_year`
- (corresponding variables for `ed_`)

**Purpose**  
These variables describe when movement events start and end. They capture both exact timestamps and derived calendar-based attributes, enabling analysis of temporal patterns in equipment movements.

In [ ]:
sd = pd.to_datetime(df["sd_date_string"], format="%d-%m-%Y")
ed = pd.to_datetime(df["ed_date_string"], format="%d-%m-%Y")

(sd <= ed).value_counts()

A consistency check confirms that start times precede or equal end times for all observations, indicating valid temporal ordering.

In [ ]:
df["st_time"].head()

In [ ]:
df["st_time"].str.len().value_counts()

Time variables are stored as strings in HH.MM.SS format.

In [ ]:
df["st_time"].str[-2:].value_counts()

Temporal variables are recorded at minute-level granularity, as seconds are consistently equal to zero across all observations.

## 6. Location variables

**Variables**
- `sl_building_global_name`
- `sl_floor_global_name`
- `sl_building_local_name`
- `sl_floor_local_name`
- `sl_locality_local_name`
- `sl_room_local_name`
- `sl_section_local_name`
- (corresponding variables for `el_`)

**Purpose**  
These variables describe the spatial location of equipment at both the start and end of each movement event, enabling movements to be traced across locations. Variables prefixed with `sl_` represent the start location, while variables prefixed with `el_` represent the end location of a movement.

The variables follow a hierarchical structure, including building, floor, locality, room, and section levels, allowing location data to be analyzed at different levels of spatial granularity.

In [ ]:
location_cols = [col for col in df.columns if col.startswith("sl_") or col.startswith("el_")]

df[location_cols].head()

Location variables are available at both global and local levels. Global variables provide a standardized representation of locations, often including a common prefix such as “Hospital site”, ensuring consistency across the dataset. 

### 6.1 Granularity

#### Global

In [ ]:
print("Global hierarchy")
print("Buildings:", df["sl_building_global_name"].nunique())
print("Floors:", df["sl_floor_global_name"].nunique())
print("Rooms:", df["sl_room_global_name"].nunique())
print("Sections:", df["sl_section_global_name"].nunique())

#### Local

In [ ]:
print("Local hierarchy")
print("Buildings:", df["sl_building_local_name"].nunique())
print("Floors:", df["sl_floor_local_name"].nunique())
print("Rooms:", df["sl_room_local_name"].nunique())
print("Sections:", df["sl_section_local_name"].nunique())

An inspection of location variables shows that global and local representations contain the same number of unique values across all hierarchical levels. This indicates that both representations provide an equivalent level of spatial granularity in the dataset.

Consequently, no distinction between global and local location variables is required for the analysis.

## 7. Explore data

This section explores the dataset through visualizations and simple statistical analyses to identify distributions, relationships, and initial patterns in equipment movements. The aim is to generate preliminary insights that inform subsequent data preparation and modeling steps.

### 7.1 Movement patterns

#### Distribution of Movement Metrics
This section examines the distribution of key movement-related measures, including movement distance, movement duration, and stay duration.

In [ ]:
import matplotlib.pyplot as plt

cols = ["distance_mtr", "duration_sec", "stay_duration_sec"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(cols):
    axes[i].hist(df[col], bins=100)
    
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

The distributions are highly right-skewed, with most observations concentrated at low values and a small number of extreme values stretching the scale. 

A log transformation is applied to improve visibility by compressing large values and spreading out smaller ones, allowing the underlying distribution patterns to be more clearly observed.

In [ ]:
import matplotlib.pyplot as plt

# Log-scale (important for skewed data)
cols = ["distance_mtr", "duration_sec"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

viridis_color = plt.cm.viridis(0.4)  # fast farve fra viridis

for i, col in enumerate(cols):
    values = df[col]
    values = values[values > 0]  # log requires > 0
    
    axes[i].hist(values, bins=100, color=viridis_color, edgecolor="none")
    axes[i].set_xscale("log")
    axes[i].set_title(f"{col} (log scale)")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

The distributions are highly right-skewed, with most movements being short and a few very large values.

Movement distance and duration show considerable variability, indicating a wide range of movement behaviour across observations.

Stay duration is extremely skewed, with most observations at low values and a small number of very large values.

#### Movement frequency per device

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Movement frequency per device
movement_count = df.groupby("object_key").size()

# ECDF
x = np.sort(movement_count)
y = np.arange(1, len(x) + 1) / len(x)

# Plot
plt.plot(x, y)
plt.xlabel("Number of movements per device")
plt.ylabel("Cumulative proportion of devices")
plt.title("ECDF - Movement frequency per device")
plt.show()

The ECDF shows that most volumetric pumps have a relatively low number of movements, with a steep increase at lower values.

The curve gradually flattens, indicating that only a small proportion of devices account for a high number of movements, reflecting uneven usage across devices.

#### Movement ratio

In [ ]:
ratio = df["duration_sec"] / (df["duration_sec"] + df["stay_duration_sec"])

In [ ]:
pd.reset_option('display.float_format')

In [ ]:
ratio.describe()

The movement ratio is generally low, with a median close to zero, indicating that devices spend most of their time stationary.

The distribution is highly skewed, with only a small proportion of observations showing higher movement activity.

#### Stay duration based on week day

In [ ]:
df.groupby("sd_day_of_week_name")["stay_duration_sec"].mean()

Average stay duration is higher on weekdays than weekends, indicating longer stationary periods during weekdays.

#### High-activity vs. low-activity devices

In [ ]:
movement_count = df.groupby("object_key").size()

high_activity = movement_count[movement_count > movement_count.median()]
low_activity = movement_count[movement_count <= movement_count.median()]

In [ ]:
plt.bar(["High activity", "Low activity"], [len(high_activity), len(low_activity)])
plt.title("Number of devices by activity level")
plt.ylabel("Number of devices")
plt.show()

#### Stay duration by device activity level

In [ ]:
import matplotlib.pyplot as plt

# Få keys
high_keys = high_activity.index
low_keys = low_activity.index

# Lav activity group kolonne
df["activity_group"] = df["object_key"].apply(
    lambda x: "High activity" if x in high_keys else "Low activity"
)

# Split data
data = [
    df[df["activity_group"] == "High activity"]["stay_duration_sec"],
    df[df["activity_group"] == "Low activity"]["stay_duration_sec"]
]

# Viridis farver
colors = [plt.cm.viridis(0.25), plt.cm.viridis(0.75)]

fig, ax = plt.subplots()

box = ax.boxplot(
    data,
    patch_artist=True,
    tick_labels=["High activity", "Low activity"],
    flierprops=dict(
        marker='o',
        markersize=4,
        linestyle='none',
        markerfacecolor='white'  # hvid midte
    )
)

# Farv bokse + outliers (kant)
for i, (patch, color) in enumerate(zip(box["boxes"], colors)):
    patch.set_facecolor(color)
    box["fliers"][i].set_markeredgecolor(color)  # farvet ring

# Labels
ax.set_title("Stay duration by device activity level")
ax.set_xlabel("Activity level")
ax.set_ylabel("Stay duration (seconds)")

plt.tight_layout()
plt.show()

The figure shows the distribution of stay duration across high- and low-activity devices. High-activity devices tend to have shorter intervals between movements, while low-activity devices remain stationary for longer periods. The distributions are highly skewed, with several extreme values in both groups.

In [ ]:
df["hour"] = df["st_time"].str.slice(0,2).astype(int)

high_keys = high_activity.index
low_keys = low_activity.index

df_high = df[df["object_key"].isin(high_keys)]
df_low = df[df["object_key"].isin(low_keys)]

df_high.groupby("hour").size().plot(label="High activity")
df_low.groupby("hour").size().plot(label="Low activity")

plt.legend()
plt.title("Activity over time")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Viridis colors
colors = plt.cm.viridis([0.2, 0.8])

plt.figure(figsize=(9,5))

df_high.groupby("hour").size().plot(
    label="High activity",
    linewidth=2.5,
    color=colors[0]
)

df_low.groupby("hour").size().plot(
    label="Low activity",
    linewidth=2.5,
    color=colors[1]
)

plt.xlabel("Hour of day")
plt.ylabel("Recorded movement events")
plt.title("Hourly movement patterns across high- and low-activity devices")

plt.xticks(range(24))
plt.legend(frameon=False)

plt.tight_layout()

# Save figure
plt.savefig(
    "hourly_activity_by_device_level.png",
    dpi=300,
    bbox_inches="tight",
    transparent=False
)

plt.show()

High-activity devices show a pronounced peak during daytime hours, while low-activity devices exhibit consistently low activity throughout the day.

This indicates that a subset of devices drives the majority of movement activity, particularly during peak daytime periods.

In [ ]:
print(df_high["stay_duration_sec"].median())
print(df_low["stay_duration_sec"].median())

Low-activity devices have a substantially higher median stay duration than high-activity devices, indicating longer idle periods and lower utilization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sort values (descending)
high_loc = df_high["sl_floor_local_name"].value_counts().sort_values(ascending=False)
low_loc = df_low["sl_floor_local_name"].value_counts().sort_values(ascending=False)

# Normalize to cumulative share
high_cum = np.cumsum(high_loc) / high_loc.sum()
low_cum = np.cumsum(low_loc) / low_loc.sum()

# X-axis: share of locations
high_x = np.arange(1, len(high_cum)+1) / len(high_cum)
low_x = np.arange(1, len(low_cum)+1) / len(low_cum)

# Viridis colors
color_high = plt.cm.viridis(0.25)  # mørkere
color_low = plt.cm.viridis(0.75)   # lysere

# Plot
plt.plot(high_x, high_cum, label="High activity", color=color_high, linewidth=2)
plt.plot(low_x, low_cum, label="Low activity", color=color_low, linewidth=2)

plt.xlabel("Cumulative share of locations")
plt.ylabel("Cumulative share of movements")
plt.title("Concentration of movements across locations")
plt.legend()

plt.tight_layout()
plt.show()

High-activity devices exhibit a strong concentration of movements in a small number of locations, as indicated by the steep initial increase in the curve.

In contrast, low-activity devices are more evenly distributed across locations, reflected by the more gradual slope.

### 7.2 Temporal patterns

#### Distribution of movement events across hours of the day

In [ ]:
hours = pd.to_datetime(df["st_time"], format="%H.%M.%S").dt.hour
hours.value_counts().sort_index()

In [ ]:
hours.value_counts().sort_index().plot(kind="bar")

Movement events are concentrated during daytime hours, with peak activity observed around midday and early afternoon, while activity is lower during nighttime.

#### Device activity: day vs night comparison

In [ ]:
df["hour"] = df["st_time"].str.slice(0,2).astype(int)

df["time_period"] = df["hour"].apply(
    lambda x: "Day" if 7 <= x < 19 else "Night"
)

movement_day_night = df.groupby(["object_key", "time_period"]).size().unstack(fill_value=0)

movement_day_night.sum().plot(kind="bar")
plt.title("Total movements: Day vs Night")
plt.ylabel("Number of movements")
plt.show()

Movement activity is substantially higher during daytime, with lower but consistent activity observed at night.

In [ ]:
movement_day_night.plot(kind="scatter", x="Day", y="Night")
plt.title("Device activity: Day vs Night")
plt.xlabel("Day movements")
plt.ylabel("Night movements")
plt.show()

Devices with higher daytime activity also tend to exhibit higher nighttime activity, indicating a positive relationship between day and night usage.

#### Average movement duration by hour of day

In [ ]:
df.groupby("hour")["duration_sec"].mean().plot()

Average movement duration varies throughout the day, with a noticeable peak in the morning hours and relatively stable levels during daytime, followed by a decline in the evening.

### 7.3 Location patterns

#### Distribution across locations

In [ ]:
df["sl_room_local_name"].value_counts().head(30).plot(kind="barh")
plt.title("Top 10 most frequent locations")
plt.xlabel("Number of observations")
plt.show()

In [ ]:
pd.set_option("display.max_rows", None)

print(df["sl_room_local_name"].value_counts().to_string())

In [ ]:
df["sl_floor_local_name"].value_counts().head(100) # distribution across floors

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Top 15 floors
top_floors = df["sl_floor_local_name"].value_counts().head(15).sort_values()

plt.figure(figsize=(10, 6))

# Viridis farver (fra mørk → lys)
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(top_floors)))

# Plot
bars = plt.barh(top_floors.index, top_floors.values, color=colors)

plt.title("Distribution of movement events across floors")
plt.xlabel("Number of movements")
plt.ylabel("")

# Fjern top/right border
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

# Labels på barer
for i, v in enumerate(top_floors.values):
    plt.text(v + 50, i, str(v), va='center')

plt.tight_layout()
plt.show()

Device activity is unevenly distributed across locations and floors, with a few corridors and storage areas accounting for a large share of observations.

At the floor level, activity is concentrated on a limited number of floors (e.g. B.02, D.04, D.05), indicating clustered device usage within specific areas.

#### Device presence (aggregation)

In [ ]:
top_locations = df.groupby("sl_room_local_name")["stay_duration_sec"].sum() \
                  .sort_values(ascending=False) \
                  .head(20)

top_locations.plot(kind="barh")
plt.title("Top 20 locations by total stay duration")
plt.xlabel("Total stay duration (sec)")
plt.show()

Total stay duration is highly concentrated in a small number of locations, primarily storage, depot, and repair areas.

Compared to the frequency-based analysis, which highlighted corridors and transit locations, this indicates that devices spend most of their time stationary in dedicated storage and service locations rather than in transit areas.

This highlights a clear distinction between movement-heavy locations and locations associated with prolonged device presence.

#### Movement between locations

In [ ]:
df.groupby(["sl_room_local_name", "el_room_local_name"]) \
  .size() \
  .sort_values(ascending=False) \
  .head(30)

Movement patterns are dominated by transitions within the same location, indicating that devices often remain within a single room or area.

Recurring bidirectional flows are also observed between specific locations, particularly between corridors and storage areas, suggesting structured and repeated movement patterns.

#### Movement based on locations

In [ ]:
df.groupby("sl_floor_local_name")["duration_sec"].mean().sort_values(ascending=False).head(20)

Average movement duration varies across floors, with some floors exhibiting notably longer movements than others.

#### Location types (room-level classification)

To further interpret location patterns, room-level labels are grouped into broader location types based on their semantic meaning. This classification distinguishes between storage/service locations (e.g., storage, depot, repair), transit areas (e.g., corridors, elevators), and other locations.

In [ ]:
def categorize_room(x):
    if not isinstance(x, str):
        return "Unknown"
    
    x_lower = x.lower()
    
    if any(word in x_lower for word in ["storage", "depot", "repair", "recovery", "equipment", "logistiek"]):
        return "Storage/Service"
    elif any(word in x_lower for word in ["corridor", "elevator", "stairwell"]):
        return "Transit"
    else:
        return "Clinical/Other"

df["location_type"] = df["sl_room_local_name"].apply(categorize_room)

In [ ]:
df["location_type"].value_counts()

Room-level classification shows that a substantial share of observations is categorized as unknown due to missing location information. Among the observed locations, storage and service areas and transit locations account for a large share of events, while fewer observations are associated with other location types.

This indicates that room-level data provides useful indications of location types, but should be interpreted with caution due to missing values.

In [ ]:
df.groupby("location_type")["stay_duration_sec"].sum()

A large share of total stay duration is associated with unknown locations due to missing room-level data.

Among the observed locations, storage and service areas account for the highest share of stationary time, followed by transit and other location

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Movement distribution ---
movement_by_type = df["location_type"].value_counts()

# --- Stay duration (convert to hours) ---
stay_by_type = df.groupby("location_type")["stay_duration_sec"].sum() / 3600

# --- Combine into table ---
table_A4 = pd.DataFrame({
    "Movement count": movement_by_type,
    "Total stay duration (hours)": stay_by_type
}).fillna(0)

# --- Order categories ---
order = ["Storage/Service", "Transit", "Clinical/Other", "Unknown"]
table_A4 = table_A4.reindex(order)

# --- Clean formatting ---
table_A4["Movement count"] = table_A4["Movement count"].astype(int)
table_A4["Total stay duration (hours)"] = table_A4["Total stay duration (hours)"].round(0).astype(int)

# --- Create table plot ---
fig, ax = plt.subplots(figsize=(5.5, 2.5))
ax.axis('off')

table = ax.table(
    cellText=table_A4.values,
    colLabels=table_A4.columns,
    rowLabels=table_A4.index,
    loc='center'
)

# --- Styling ---
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.2)

# Center text
for key, cell in table.get_celld().items():
    cell.set_text_props(ha='center')

# --- Save as JPEG ---
plt.savefig("table_A4.jpeg", bbox_inches='tight', dpi=300)
plt.close()

#### Movement vs. stay duration across locations

To further examine how activity is distributed across locations, movement frequency and total stay duration are compared at floor level.

In [ ]:
movement = df["sl_floor_local_name"].value_counts()
stay = df.groupby("sl_floor_local_name")["stay_duration_sec"].sum()

combined = pd.DataFrame({
    "movement_count": movement,
    "stay_duration": stay
}).fillna(0)

combined.sort_values("movement_count", ascending=False).head(10)

The results show that high-activity floors also account for substantial stay duration. This indicates that locations often function as both movement hubs and areas of prolonged equipment presence, rather than reflecting a clear separation between transit and storage.

In [ ]:
!pip install adjustText

In [ ]:
from adjustText import adjust_text
import matplotlib.pyplot as plt

plot_df = combined.reset_index()

plt.figure(figsize=(8,6))

# Viridis farve
color = plt.cm.viridis(0.4)

plt.scatter(
    plot_df["movement_count"],
    plot_df["stay_duration"],
    color=color,
    alpha=0.8
)

# Only pick top floors
top = plot_df.sort_values("movement_count", ascending=False).head(8)

texts = []
for _, row in top.iterrows():
    texts.append(
        plt.text(
            row["movement_count"],
            row["stay_duration"],
            row["sl_floor_local_name"],
            fontsize=8
        )
    )

# Adjust text to avoid overlap
adjust_text(texts)

plt.xlabel("Number of movements")
plt.ylabel("Total stay duration (sec)")
plt.title("Movement activity vs. stay duration across floors")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Create table ---
movement = df["sl_floor_local_name"].value_counts()
stay = df.groupby("sl_floor_local_name")["stay_duration_sec"].sum()

combined = pd.DataFrame({
    "movement_count": movement,
    "stay_duration_sec": stay
}).fillna(0)

# --- Convert to hours ---
combined["stay_duration_hours"] = combined["stay_duration_sec"] / 3600

# --- Select and rename columns ---
combined_sorted = combined.sort_values("movement_count", ascending=False)

table_data = combined_sorted[["movement_count", "stay_duration_hours"]].head(15)

table_data = table_data.rename(columns={
    "movement_count": "Movement count",
    "stay_duration_hours": "Total stay duration (hours)"
})

# --- Round values for presentation ---
table_data = table_data.round(1)

# --- Display table ---
table_data

# --- (Optional) Export as image ---
fig, ax = plt.subplots(figsize=(8,3))
ax.axis('off')

table = ax.table(
    cellText=table_data.values,
    colLabels=table_data.columns,
    rowLabels=table_data.index,
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.2)

plt.savefig("table_A3.jpeg", bbox_inches='tight', dpi=300)
plt.close()

#### Lorenz curve of movement concentration across devices

To further examine how movement activity is distributed across devices, a Lorenz curve is used to visualize concentration in movement events.

If movement activity were evenly distributed, the curve would follow the diagonal reference line. Deviation below this line indicates that a smaller subset of devices accounts for a disproportionate share of total movement events.

This helps assess whether equipment activity is concentrated among relatively few devices, which may indicate uneven operational distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Movement frequency per device
device_activity = df.groupby("object_key").size().sort_values()

# Cumulative shares
cum_devices = np.arange(1, len(device_activity) + 1) / len(device_activity)
cum_activity = device_activity.cumsum() / device_activity.sum()

# Viridis colors
color_lilla = plt.cm.viridis(0.1)   # lilla
color_gul = plt.cm.viridis(0.9)     # gul

# Plot Lorenz curve
plt.figure(figsize=(6,6))

plt.plot(cum_devices, cum_activity, 
         label="Observed distribution", 
         color=color_lilla, linewidth=2)

plt.plot([0,1], [0,1], 
         linestyle="--", 
         label="Perfect equality", 
         color=color_gul, linewidth=2)

plt.xlabel("Cumulative share of devices")
plt.ylabel("Cumulative share of movement events")
plt.title("Lorenz curve of movement concentration across devices")
plt.legend()

plt.tight_layout()
plt.show()

### 7.4 Conclusion 

Overall, the exploratory analysis reveals that movement behaviour is highly uneven across time, locations, and individual devices.

Most movements are short and infrequent, with distributions characterized by strong right-skewness and substantial variation. A small subset of devices accounts for a large share of movement activity, while many devices exhibit low activity and long idle periods.

Temporal patterns show that movement activity is concentrated during daytime hours, with reduced but consistent activity at night. Additionally, differences between high- and low-activity devices highlight heterogeneous usage patterns across devices.

Location-based analysis indicates that movement events are concentrated in corridors and transit areas, whereas prolonged device presence is associated with storage, depot, and service locations. Furthermore, high-activity devices are concentrated in fewer locations, while low-activity devices are more dispersed across the hospital.

These findings suggest imbalanced utilization of devices and provide a foundation for subsequent data preparation and predictive modelling of device usage.

In [ ]:
# Drop variables created in Data exploration
df = df.drop(columns=["hour", "time_period"])

## 8. Verify data quality

This step follows CRISP-DM (2.4) and focuses on assessing the quality of the dataset. The data is evaluated in terms of completeness, correctness, and consistency to identify potential issues that may affect subsequent analysis and modelling.

### 8.1 Completeness

In [ ]:
df.isna().sum().sort_values(ascending=False)

**Insight:**  
- Missing values are primarily concentrated in location-related variables (room, building, section, and floor)  
- No missing values are observed in key analytical variables (e.g., distance, duration, stay duration)  
- The missingness appears to be structural, as not all movements are associated with specific locations  
- Overall, the dataset is considered sufficiently complete for further analysis  

#### Investigating missingness in location variables

In [ ]:
location_cols = [col for col in df.columns if "room" in col or "building" in col or "section" in col or "floor" in col]

missing = df[location_cols].isna().mean().sort_values(ascending=False)

plt.figure()
missing.plot(kind="bar")

plt.xticks(rotation=90)
plt.ylabel("Missing ratio")
plt.title("Missing values in location variables")

plt.tight_layout()
plt.show()

The analysis of missing values reveals a clear pattern across location-related variables. Missingness is highest at the most granular level, with over 50% of room-level information absent, while building- and section-level variables show moderate levels of missingness around 30–35%. In contrast, floor-level information is almost complete, with only around 3–4% missing values.

Importantly, no meaningful difference is observed between local and global location variables, as they exhibit identical missingness patterns across all levels. This indicates that both representations are derived from the same underlying data and are equally affected by missing values.

This hierarchical pattern suggests that location data becomes more complete at higher levels of aggregation. In other words, while precise room information is often unavailable, broader spatial context such as floor or building is more consistently recorded.

The missingness therefore appears to be systematic rather than random, likely reflecting limitations in how detailed location data is captured within the system. As a result, analyses at finer spatial granularity (e.g., room-level) should be interpreted with caution, whereas higher-level analyses (e.g., floor-level) are considered more reliable.

#### Asymmetric missingness in room locations

In [ ]:
df[["sl_room_local_name", "el_room_local_name"]].isna().value_counts()

In [ ]:
df[
    df["sl_room_local_name"].isna() ^ df["el_room_local_name"].isna()
][["sl_room_local_name", "el_room_local_name"]].head(10)

Missing room information can occur in either the start or end location independently, confirming that missingness is not consistently paired.

#### Missingness by movement complexity

In [ ]:
df.groupby("segment_count")[location_cols].apply(lambda x: x.isna().mean())

Missing location data shows some variation with segment count, but no clear monotonic pattern emerges, suggesting that movement complexity alone does not systematically explain missingness, although simpler movements (e.g., single segments) tend to have slightly more complete location information.

#### Missingness by zero-distance movements


In [ ]:
df["zero_distance"] = df["distance_mtr"] == 0

df.groupby("zero_distance")[location_cols].apply(lambda x: x.isna().mean())

Missing location data is consistently lower for zero-distance movements compared to movements with distance, particularly at room and section level. This suggests that stationary events are more likely to be associated with well-defined locations, whereas moving events more often lack precise spatial information.

#### Completeness conclusion
Overall, the dataset is considered sufficiently complete for analysis, as all key variables related to movement (e.g., distance, duration, and stay duration) are fully populated. Missing values are primarily confined to location-related variables and follow a systematic pattern, which does not compromise the overall usability of the data.

### 8.2 Correctness

In [ ]:
df.describe()

The numerical variables appear generally valid, with no indication of impossible values such as negative distances or durations. However, the distributions exhibit strong right-skewness and substantial variation, with extreme values observed across distance, duration, and stay duration. These high maximum values are considered plausible given the context, reflecting occasional long movements or extended inactivity periods rather than data errors.

#### Outlier inspection

In [ ]:
df[["distance_mtr", "duration_sec", "stay_duration_sec","segment_count", "distance_floor"]].describe()

The numerical variables show large variation, with most movements being short and simple, but a few cases standing out with very high distances, durations, and segment counts. The maximum floor distance also seems unusually high. These values are likely rare but realistic events rather than errors. 

In [ ]:
df[["distance_mtr", "duration_sec", "stay_duration_sec", "segment_count", "distance_floor"]].corr()

The correlation analysis shows that most variables are only weakly related, with distance, duration, and stay duration largely independent of each other. However, a strong relationship is observed between duration and segment count, indicating that longer movements typically involve more segments. Distance shows only a weak correlation with both duration and segment count, suggesting that long distances are not necessarily associated with longer or more complex movements. Overall, extreme values do not appear to be driven by a single variable, but rather represent different types of rare events.

#### Logical consistency checks

In [ ]:
# Check zero distance with positive duration (possible stationary events)
df[(df["distance_mtr"] == 0) & (df["duration_sec"] > 0)]

Inspection of cases with zero distance and positive duration shows that these observations correspond to stationary events where devices remain at the same location over time. This confirms that such cases are valid and do not represent data errors.

In [ ]:
# Checks that end time occurs after start time (no negative durations).
(df["ed_day_of_year"] - df["sd_day_of_year"]).describe()

No negative time differences are observed, confirming consistent temporal ordering of events.

#### Correctness conclusion

Overall, the data appears valid with no obvious errors. Extreme values are present but reflect realistic variation rather than inconsistencies, and basic logical checks confirm that the dataset is internally consistent.

### 8.3 Consistency

In [ ]:
df.duplicated().sum()

In [ ]:
# Ensure end time is not before start time (temporal consistency)
(df["ed_day_of_year"] - df["sd_day_of_year"]).min()

In [ ]:
# Same location but non-zero distance?
df[(df["sl_room_local_name"] == df["el_room_local_name"]) & (df["distance_mtr"] > 0)].count()

A notable number of observations show movement (non-zero distance) despite identical start and end room locations. This suggests that distance may capture internal movement within the same room or reflect measurement granularity, rather than indicating inconsistency in the data.

In [ ]:
df[
    (df["distance_mtr"] == 0) & 
    (df["sl_room_local_name"] != df["el_room_local_name"])
][["sl_room_local_name", "el_room_local_name", "distance_mtr"]].head(10)

In [ ]:
df[
    (df["distance_mtr"] == 0) & 
    (df["sl_room_local_name"] != df["el_room_local_name"])
].shape[0]

Zero-distance movements between different locations likely reflect limitations in spatial resolution or measurement granularity, rather than clear data errors.

In [ ]:
room = df.groupby("sl_room_local_name")["distance_mtr"].agg(
    zero_dist=lambda x: (x == 0).any(),
    positive_dist=lambda x: (x > 0).any()
).query("zero_dist and positive_dist").index[0]

df[df["sl_room_local_name"] == room][
    ["sl_room_local_name", "el_room_local_name", "distance_mtr"]
].head(30)

Observations from the same start and end location exhibit both zero and positive distances, indicating that distance captures internal movement or measurement variation rather than strictly representing transitions between distinct locations.

#### Distance variance

In [ ]:
same_loc = df[df["sl_room_local_name"] == df["el_room_local_name"]]

same_loc["distance_mtr"].describe()

Distances within the same location are often substantial, with a median of 27 meters and extreme values exceeding 600 meters. This indicates that the distance metric captures accumulated or internal movement over time rather than strictly representing transitions between distinct locations.

In [ ]:
(same_loc["distance_mtr"] > 0).mean()

In [ ]:
# Identify locations where same start and end location has distance > 0
rooms_with_internal_movement = df[
    (df["sl_room_local_name"] == df["el_room_local_name"]) &
    (df["distance_mtr"] > 0)
]["sl_room_local_name"].unique()

# Check if these locations also have distance = 0 when moving to a different location
df[
    (df["sl_room_local_name"].isin(rooms_with_internal_movement)) &
    (df["distance_mtr"] == 0) &
    (df["sl_room_local_name"] != df["el_room_local_name"])
][["sl_room_local_name", "el_room_local_name", "distance_mtr"]].head(10)

Locations that exhibit internal movement with positive distance also show zero-distance transitions to other locations. This inconsistency indicates that the distance metric is not strictly tied to changes in location, but is influenced by how movement is captured or aggregated in the system.

#### 8.3 Conclusion 
Overall, the dataset demonstrates a high degree of internal consistency. Data types have been standardized across variables, no duplicate records are identified, and temporal variables follow a logically consistent order, with no instances of end times occurring before start times.

Further consistency checks reveal that relationships between location and distance variables are not always strictly aligned. In particular, observations show that movements within the same location can have substantial positive distances, while transitions between different locations may occasionally have zero distance. This indicates that distance measures are influenced by how movement is captured or aggregated in the system, rather than strictly representing physical displacement between locations.

These inconsistencies are therefore interpreted as characteristics of the measurement process rather than data errors, and do not undermine the overall integrity of the dataset.

### 8.4 Data quality implications

The identified data quality issues are not critical but have implications for the analysis. Missing location data is primarily structural and should be considered when interpreting location-based patterns, particularly at finer levels of granularity. 

Furthermore, inconsistencies between distance and location variables indicate that distance measures reflect movement characteristics rather than exact spatial transitions. As a result, distance is treated as a relative rather than absolute measure.

Overall, the dataset is deemed suitable for further analysis, with appropriate handling of skewed distributions and awareness of limitations in location precision.

The dataset is considered relevant for the analysis, as it captures detailed movement events of volumetric pumps across time and locations, enabling analysis of utilization patterns.